In [0]:
%python

from pyspark.sql.functions import *
from pyspark.sql.types import *

### FETCHING PARAMETERS

In [0]:
%python

import json

# SOURCE TABLE
source_table = dbutils.widgets.get("source_table")

# SOURCE SCHEMA
source_schema = dbutils.widgets.get("source_schema")

# TARGET SCHEMA
target_schema = dbutils.widgets.get("target_schema")

# TARGET TABLE
target_table = dbutils.widgets.get("target_table")

# KEY COLUMNS
key_cols_list = json.loads(dbutils.widgets.get("key_cols"))       

# CDC COLUMN
cdc_col = dbutils.widgets.get("cdc_col")

# BACK-DATED REFRESH
backdated_refresh = dbutils.widgets.get("backdated_refresh")

# SURROGATE KEY
surrogate_key = dbutils.widgets.get("surrogate_key")

### Incremental Data Ingestion

#### Last Load Date

In [0]:
%python

from datetime import datetime

INITIAL_LOAD = datetime(1900, 1, 1, 0, 0, 0)

if backdated_refresh:

    last_load = backdated_refresh

else:
    target = (f"flights_project.{target_schema}.{target_table}")

    if spark.catalog.tableExists(target):
        last_load = spark.sql(f"""
            SELECT MAX({cdc_col})
            FROM {target}
        """).first()[0]             # retrieves the single result without collecting a list.

        # Handle an existing but empty target table
        if last_load is None:
            last_load = INITIAL_LOAD
    else:

        last_load = INITIAL_LOAD

print(f"Last load timestamp: {last_load}")

In [0]:
%python

df_source = spark.sql(f"""
                SELECT *
                FROM flights_project.{source_schema}.{source_table}
                WHERE {cdc_col} > '{last_load}'""")

# Create source temporary view                
df_source.createOrReplaceTempView("src")

In [0]:
%python

"""
- Builds the target DataFrame — existing key columns + surrogate key + audit dates if the table exists.
- Or a pseudo/empty schema if table not exists. Registers a trg temp view
"""

if spark.catalog.tableExists(f"flights_project.{target_schema}.{target_table}"):

    # Key Columns String For Incremental
    key_cols_string_incremental = ", ".join(key_cols_list)

    df_target = spark.sql(f"""
                        SELECT
                            {key_cols_string_incremental},
                            {surrogate_key},
                            create_date,
                            update_date   
                        FROM flights_project.{target_schema}.{target_table}""")
    
# Empty target -> Pseudo table
else:
    # Key Columns String For Initial
    key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
    key_cols_string_init = ", ".join(key_cols_string_init)
    df_target = spark.sql(f"""
                        SELECT
                            {key_cols_string_init},
                            CAST('0' AS INT) AS {surrogate_key},
                            CAST("1900-01-01 00:00:00" AS TIMESTAMP) AS create_date,
                            CAST("1900-01-01 00:00:00" AS TIMESTAMP) AS update_date
                        FROM range(0)           
                        WHERE 1 = 0""")
# Create target temporary view 
df_target.createOrReplaceTempView("trg")

### JOIN CONDITION

In [0]:
%python

join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

df_join = spark.sql(f"""
                    SELECT 
                        src.*,
                        trg.{surrogate_key},
                        trg.create_date,
                        trg.update_date
                    FROM src
                    LEFT JOIN trg
                    ON {join_condition}""")

### Old vs. New Records

In [0]:
%python

df_old = df_join.filter(col(f"{surrogate_key}").isNotNull())        # old records already have surrogate key in the target table
df_new = df_join.filter(col(f"{surrogate_key}").isNull())           # new records don't have surrogate key in the target table 

### Preparing df_old

In [0]:
%python

df_old_enrich = df_old.withColumn('update_date', current_timestamp())

### Preparing df_new

In [0]:
%python

if spark.catalog.tableExists(f"flights_project.{target_schema}.{target_table}"):
   
    max_surrogate_key = spark.sql(f"""
                            SELECT
                                MAX({surrogate_key}) 
                            FROM flights_project.{target_schema}.{target_table}""").collect()[0][0]

    df_new_enrich = (df_new.withColumn(f"{surrogate_key}", lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())
                        .withColumn('create_date', current_timestamp())
                        .withColumn('update_date', current_timestamp()))

else:
    max_surrogate_key = 0
    df_new_enrich = (df_new.withColumn(f"{surrogate_key}", lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())
                        .withColumn('create_date', current_timestamp())
                        .withColumn('update_date', current_timestamp()))

### Union OLD and NEW records

In [0]:
%python

df_union = df_old_enrich.unionByName(df_new_enrich)

### UPSERT

In [0]:
%python

from delta.tables import DeltaTable

if spark.catalog.tableExists(f"flights_project.{target_schema}.{target_table}"):
    
    dlt_object = DeltaTable.forName(spark, f"flights_project.{target_schema}.{target_table}")
    (dlt_object.alias("trg")
            .merge(df_union.alias("src"),
                    f"trg.{surrogate_key} = src.{surrogate_key}")\
            .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
            .whenNotMatchedInsertAll()\
            .execute())

else:
    df_union.write.format('delta')\
                .mode("append")\
                .saveAsTable(f"flights_project.{target_schema}.{target_table}")